# LeetCode 624 — Maximum Distance in Arrays bằng Python

Link đề: https://leetcode.com/problems/maximum-distance-in-arrays/ · Độ khó: **Medium** · Chủ đề: Greedy, Array

Skeleton LeetCode:

```python
class Solution:
    def maxDistance(self, arrays: List[List[int]]) -> int:
        ...
```

Ghi chú: bài có **đáp án duy nhất** (một số nguyên) nên so sánh bằng `==` được. Điểm khó nằm ở chỗ **hai số phải lấy từ hai mảng khác nhau**: câu "lấy max trừ min của tất cả" nghe hợp lý nhưng sai.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: có `m` mảng đã sắp xếp tăng dần; chọn **mỗi mảng một số** (hai mảng **khác nhau**) sao cho `|a - b|` lớn nhất.
2. Nắm các công cụ Python: truy cập `arr[0]` và `arr[-1]`, `abs`, `min`/`max` có nhiều đối số, vòng lặp `for arr in arrays[1:]`, `itertools.combinations`, `heapq.nsmallest/nlargest`, tạo dữ liệu ngẫu nhiên có seed bằng `random.Random(seed)`.
3. Đi từ **vét cạn mọi cặp số** → **chỉ xét hai đầu mút của mỗi mảng** → **duyệt một lượt giữ min/max đã thấy** → **giữ hai giá trị nhỏ nhất / lớn nhất kèm chỉ số mảng**.
4. Hiểu mấu chốt: vì mảng đã sắp xếp, số nhỏ nhất của mảng là `arr[0]`, lớn nhất là `arr[-1]`; đáp án luôn là `max` của `(đầu mút phải của mảng j) − (đầu mút trái của mảng i)` với `i ≠ j`.
5. Tránh các lỗi kinh điển: lấy `max toàn cục − min toàn cục` (có thể cùng một mảng), cập nhật `mi/mx` **trước** khi so sánh, chỉ xét một chiều, hoặc chỉ so sánh hai mảng liền kề.

## 2. Đề bài & ví dụ

Cho `m` mảng `arrays`, mỗi mảng đã được sắp xếp **tăng dần**. Bạn chọn một số nguyên từ mỗi trong **hai mảng khác nhau** rồi tính **khoảng cách** giữa hai số `a`, `b` là `|a - b|`. Trả về khoảng cách lớn nhất có thể.

**Ví dụ 1:**

```
Input:  arrays = [[1,2,3],[4,5],[1,2,3]]
Output: 4
```

Chọn `1` ở mảng thứ nhất (hoặc thứ ba) và `5` ở mảng thứ hai: `|1 - 5| = 4`.

**Ví dụ 2:**

```
Input:  arrays = [[1],[1]]
Output: 0
```

**Ràng buộc:**

- `m == arrays.length`, `2 <= m <= 10^5`
- `1 <= arrays[i].length <= 500`
- `-10^4 <= arrays[i][j] <= 10^4`
- `arrays[i]` được sắp xếp tăng dần.
- Tổng số phần tử của mọi mảng tối đa `10^5`.

In [1]:
import heapq
import random
import time
from itertools import combinations, product
from typing import List

ex1 = [[1, 2, 3], [4, 5], [1, 2, 3]]
ex2 = [[1], [1]]

for arrays in (ex1, ex2):
    print("m =", len(arrays), "| tổng số phần tử =", sum(len(a) for a in arrays), "|", arrays)
    print("  mỗi mảng đã tăng dần:", all(a == sorted(a) for a in arrays))

m = 3 | tổng số phần tử = 8 | [[1, 2, 3], [4, 5], [1, 2, 3]]
  mỗi mảng đã tăng dần: True
m = 2 | tổng số phần tử = 2 | [[1], [1]]
  mỗi mảng đã tăng dần: True


## 3. Kiến thức nền

### 3.1 Phần tử đầu và cuối của list

`a[0]` là phần tử đầu, `a[-1]` là phần tử cuối (chỉ số âm đếm từ cuối). Với mảng **đã sắp xếp tăng dần**, `a[0]` là **nhỏ nhất** và `a[-1]` là **lớn nhất**. Nhờ vậy ta không cần duyệt cả mảng để tìm min/max, chỉ cần nhìn hai đầu: `O(1)` mỗi mảng.

In [2]:
a = [1, 2, 3]
print(a[0], a[-1], "| min/max thật:", min(a), max(a))
b = [7]
print(b[0], b[-1], "(mảng một phần tử: đầu và cuối là cùng một số)")
print([x[0] for x in ex1], [x[-1] for x in ex1])

1 3 | min/max thật: 1 3
7 7 (mảng một phần tử: đầu và cuối là cùng một số)
[1, 4, 1] [3, 5, 3]


### 3.2 Khoảng cách `abs`, `min`, `max`

`abs(x)` là giá trị tuyệt đối. `max(a, b, c)` nhận nhiều đối số; `max(ans, x, y)` thường dùng để cập nhật đáp án. Chú ý khoảng cách luôn `>= 0` nên `ans` có thể khởi tạo bằng `0`.

In [3]:
print(abs(1 - 5), abs(5 - 1), abs(-3 - 4))
ans = 0
ans = max(ans, abs(1 - 5), abs(2 - 3))
print("ans =", ans)

4 4 7
ans = 4


### 3.3 Cặp phần tử: `itertools.combinations`

`combinations(iterable, 2)` sinh mọi cặp **không lặp và không thứ tự** (mỗi cặp một lần). Dùng để duyệt "mọi cặp hai mảng khác nhau" trong cách vét cạn.

In [4]:
for i, j in combinations(range(3), 2):
    print("cặp mảng", (i, j), ex1[i], ex1[j])
print("số cặp với m mảng:", "m*(m-1)/2 =", 3 * 2 // 2, "khi m = 3; khi m = 10^5 là", 10**5 * (10**5 - 1) // 2)

cặp mảng (0, 1) [1, 2, 3] [4, 5]
cặp mảng (0, 2) [1, 2, 3] [1, 2, 3]
cặp mảng (1, 2) [4, 5] [1, 2, 3]
số cặp với m mảng: m*(m-1)/2 = 3 khi m = 3; khi m = 10^5 là 4999950000


### 3.4 Giữ giá trị tốt nhất khi duyệt (running min / max)

Khi duyệt một dãy, ta có thể giữ biến `mi` (nhỏ nhất đã gặp) và `mx` (lớn nhất đã gặp) và cập nhật `mi = min(mi, x)`, `mx = max(mx, x)`. Khi tới phần tử mới, `mi`, `mx` chính là tóm tắt của **mọi phần tử trước nó** trong `O(1)` bộ nhớ. Đây là ý tưởng "duyệt một lượt" của bài.

In [5]:
xs = [5, 2, 8, 1, 9]
mi, mx = xs[0], xs[0]
for x in xs[1:]:
    print(f"gặp {x}: (mi, mx) trước = ({mi}, {mx})", end="  ->  ")
    mi, mx = min(mi, x), max(mx, x)
    print(f"sau = ({mi}, {mx})")

gặp 2: (mi, mx) trước = (5, 5)  ->  sau = (2, 5)
gặp 8: (mi, mx) trước = (2, 5)  ->  sau = (2, 8)
gặp 1: (mi, mx) trước = (2, 8)  ->  sau = (1, 8)
gặp 9: (mi, mx) trước = (1, 8)  ->  sau = (1, 9)


### 3.5 `heapq.nsmallest` / `nlargest`

`heapq.nsmallest(k, iterable)` trả `k` phần tử nhỏ nhất (đã sắp xếp tăng), `heapq.nlargest(k, iterable)` trả `k` lớn nhất (giảm dần). So sánh tuple `(giá trị, chỉ số)` theo giá trị trước. Ta dùng chúng ở Cách 4 để lấy **hai** ứng viên tốt nhất kèm chỉ số mảng.

In [6]:
pairs = [(first, idx) for idx, (first, last) in enumerate((a[0], a[-1]) for a in ex1)]
print("(đầu, chỉ số mảng):", pairs)
print("2 nhỏ nhất :", heapq.nsmallest(2, pairs))
print("2 lớn nhất :", heapq.nlargest(2, pairs))

(đầu, chỉ số mảng): [(1, 0), (4, 1), (1, 2)]
2 nhỏ nhất : [(1, 0), (1, 2)]
2 lớn nhất : [(4, 1), (1, 2)]


### 3.5b Dữ liệu ngẫu nhiên có seed

`random.Random(seed)` tạo một bộ sinh **riêng** có seed; cùng seed cho cùng dãy, nên test lặp lại được. Ta sẽ dùng để sinh các bộ mảng hợp lệ (tăng dần) để đối chiếu các cách.

In [7]:
def random_arrays(rng, m, max_len, lo, hi):
    """m mảng, mỗi mảng độ dài 1..max_len, giá trị trong [lo, hi], đã sắp xếp tăng dần."""
    return [sorted(rng.randint(lo, hi) for _ in range(rng.randint(1, max_len))) for _ in range(m)]


r1, r2 = random.Random(5), random.Random(5)
print(random_arrays(r1, 3, 4, -9, 9))
print(random_arrays(r2, 3, 4, -9, 9), "<- cùng seed, cùng kết quả")

[[-9, 2, 7], [-8, -6, -4, -2], [-2, 3, 6]]
[[-9, 2, 7], [-8, -6, -4, -2], [-2, 3, 6]] <- cùng seed, cùng kết quả


## 4. Ý tưởng nền tảng / trực giác

Hai số xa nhau nhất thì một số phải **rất nhỏ** và một số phải **rất lớn**. Với một mảng đã sắp xếp, số nhỏ nhất là đầu mảng và số lớn nhất là cuối mảng. Vậy chỉ cần quan tâm **đầu mút** của mỗi mảng; phần giữa không bao giờ giúp được.

Thử với ví dụ 1: `[[1,2,3],[4,5],[1,2,3]]`.

- Đầu mút trái (nhỏ nhất của từng mảng): `1, 4, 1`.
- Đầu mút phải (lớn nhất của từng mảng): `3, 5, 3`.
- Muốn xa nhất: lấy một số trái nhỏ + một số phải lớn **từ hai mảng khác nhau**. Ví dụ phải lớn nhất `5` (mảng 1) kết hợp trái nhỏ nhất `1` (mảng 0 hoặc 2): `5 - 1 = 4`.

**Cạm bẫy:** "số nhỏ nhất toàn cục" và "số lớn nhất toàn cục" có thể **cùng một mảng**, như `[[1, 5], [2, 3]]`: nhỏ nhất `1` và lớn nhất `5` đều thuộc mảng 0 → không được ghép (đề cấm). Đáp án đúng là `max(|1-3|, |5-2|) = 3`, không phải `4`.

In [8]:
case = [[1, 5], [2, 3]]
lo_all = min(x for a in case for x in a)
hi_all = max(x for a in case for x in a)
print("max - min toàn cục =", hi_all - lo_all, "(SAI: 1 và 5 cùng nằm trong mảng 0)")
print("đáp án thật        =", max(abs(1 - 3), abs(1 - 2), abs(5 - 2), abs(5 - 3)))

max - min toàn cục = 4 (SAI: 1 và 5 cùng nằm trong mảng 0)
đáp án thật        = 3


## 5. Cách 1 — Brute Force: thử mọi cặp số ở hai mảng khác nhau

**Ý tưởng:** làm đúng theo đề. Với mọi cặp mảng `(i, j)` (`i < j`), với mọi số `x` trong mảng `i` và `y` trong mảng `j`, tính `|x - y|` và giữ lớn nhất.

In [9]:
def max_distance_pairs(arrays):
    ans = 0
    for i, j in combinations(range(len(arrays)), 2):
        for x in arrays[i]:
            for y in arrays[j]:
                ans = max(ans, abs(x - y))
    return ans


print(max_distance_pairs(ex1), max_distance_pairs(ex2))
assert max_distance_pairs(ex1) == 4 and max_distance_pairs(ex2) == 0

4 0


**Độ phức tạp:** gọi `N` là tổng số phần tử. Mọi cặp số thuộc hai mảng khác nhau: `O(N²)` thời gian, `O(1)` bộ nhớ. Với `N = 10^5` là khoảng `5·10^9` phép tính: quá chậm. Nhưng vì rõ ràng đúng theo đề, ta dùng nó làm **oracle** (đáp án tham chiếu) để kiểm tra các cách còn lại.

**Nút thắt:** ta kiểm tra cả những số ở *giữa* mảng — không bao giờ giúp được vì đã có đầu mút xa hơn.

## 6. Ý tưởng thuật toán / bước đột phá

**Quan sát 1.** Với hai mảng `i ≠ j`, các cặp `(x, y)` có `x ∈ A_i`, `y ∈ A_j` cho `|x - y|` lớn nhất rơi vào một trong hai trường hợp: `y - x` lớn nhất khi `y = last_j` và `x = first_i`; `x - y` lớn nhất khi `x = last_i` và `y = first_j`. Do đó:

```
khoảng cách tối đa của (i, j) = max(last_j - first_i, last_i - first_j)
```

**Quan sát 2 (bỏ trị tuyệt đối).** Hai biểu thức trên cộng lại bằng `(last_i - first_i) + (last_j - first_j) >= 0`, nên ít nhất một trong hai `>= 0`; vậy không cần `abs`: đáp án toàn bài là

```
ans = max over i != j of ( last_j - first_i )
```

**Quan sát 3 (giữ tóm tắt).** Với mỗi mảng `j`, ta muốn `first_i` **nhỏ nhất** trong các mảng khác. Nếu duyệt theo thứ tự và chỉ nhìn các mảng *đứng trước*, ta giữ `mi = min first` và `mx = max last` của các mảng trước, rồi xét hai khả năng với mảng hiện tại: `last_j - mi` và `mx - first_j`. Hai chiều đó phủ mọi cặp `(i, j)` với `i < j`, tức là mọi cặp.

Điểm mấu chốt là **tính trước, cập nhật sau**: so sánh với `mi, mx` rồi mới gộp mảng hiện tại vào `mi, mx` — để không bao giờ ghép hai đầu mút của cùng một mảng.

In [10]:
# Kiểm chứng công thức: không cần abs, và max trên mọi cặp (i != j) của (last_j - first_i)
def max_distance_formula(arrays):
    m = len(arrays)
    return max(arrays[j][-1] - arrays[i][0] for i in range(m) for j in range(m) if i != j)


for arrays in (ex1, ex2, [[1, 5], [2, 3]]):
    print(arrays, "->", max_distance_formula(arrays), "| vét cạn:", max_distance_pairs(arrays))
    assert max_distance_formula(arrays) == max_distance_pairs(arrays)

[[1, 2, 3], [4, 5], [1, 2, 3]] -> 4 | vét cạn: 4
[[1], [1]] -> 0 | vét cạn: 0
[[1, 5], [2, 3]] -> 3 | vét cạn: 3


## 7. Các cách tiếp theo

Ngoài Cách 1 (mục 5), ta có thêm 3 cách: **Cách 2** xét mọi cặp mảng với đầu mút, **Cách 3** duyệt một lượt giữ `mi/mx` (cách chuẩn), **Cách 4** giữ hai ứng viên nhỏ nhất/lớn nhất kèm chỉ số.

### Cách 2 — Chỉ xét đầu mút, nhưng vẫn mọi cặp mảng

**Giải thích:** áp dụng quan sát 1 nhưng chưa tối ưu: với mỗi cặp mảng `(i, j)`, `i < j`, tính `max(last_j - first_i, last_i - first_j)`. Bỏ được vòng lặp theo phần tử nên độ phức tạp là `O(m²)` thay vì `O(N²)`.

In [11]:
def max_distance_endpoints(arrays):
    ans = 0
    for i, j in combinations(range(len(arrays)), 2):
        ans = max(ans, arrays[j][-1] - arrays[i][0], arrays[i][-1] - arrays[j][0])
    return ans


print(max_distance_endpoints(ex1), max_distance_endpoints(ex2))
assert max_distance_endpoints(ex1) == 4 and max_distance_endpoints(ex2) == 0

4 0


In [12]:
# Code nộp LeetCode cho Cách 2
class SolutionEndpoints:
    def maxDistance(self, arrays: List[List[int]]) -> int:
        ans = 0
        m = len(arrays)
        for i in range(m):
            for j in range(i + 1, m):
                ans = max(ans, arrays[j][-1] - arrays[i][0], arrays[i][-1] - arrays[j][0])
        return ans

**Độ phức tạp:** `O(m²)` thời gian, `O(1)` bộ nhớ. Với `m = 10^5` mảng (mỗi mảng một phần tử) vẫn khoảng `5·10^9`: chưa đạt.

### Cách 3 — Duyệt một lượt, giữ `mi` và `mx` (cách chuẩn, tham lam)

**Giải thích từng đoạn:**

1. Khởi tạo `mi = arrays[0][0]`, `mx = arrays[0][-1]` (tóm tắt của mảng đầu tiên) và `ans = 0`.
2. Với mỗi mảng `arr` còn lại (từ mảng thứ hai):
   - Cặp "đầu mút phải của `arr`" với "đầu mút trái nhỏ nhất ở mảng trước": `arr[-1] - mi`.
   - Cặp "đầu mút trái của `arr`" với "đầu mút phải lớn nhất ở mảng trước": `mx - arr[0]`.
   - Cập nhật `ans` bằng giá trị lớn nhất trong hai (kết quả luôn đúng với dấu vì `ans >= 0`; dùng `abs` cho an toàn cũng được).
3. **Sau đó** mới gộp: `mi = min(mi, arr[0])`, `mx = max(mx, arr[-1])`.
4. Trả `ans`.

Chữ "tham lam" nằm ở chỗ: với mỗi mảng ta chỉ cần đối chiếu với *tóm tắt tốt nhất* của các mảng trước thay vì từng mảng.

In [13]:
def max_distance_scan(arrays):
    mi, mx = arrays[0][0], arrays[0][-1]
    ans = 0
    for arr in arrays[1:]:
        ans = max(ans, arr[-1] - mi, mx - arr[0])    # so sánh TRƯỚC
        mi = min(mi, arr[0])                          # rồi mới cập nhật
        mx = max(mx, arr[-1])
    return ans


print(max_distance_scan(ex1), max_distance_scan(ex2))
assert max_distance_scan(ex1) == 4 and max_distance_scan(ex2) == 0

4 0


In [14]:
# Code nộp LeetCode cho Cách 3 (đáp án chính)
class SolutionScan:
    def maxDistance(self, arrays: List[List[int]]) -> int:
        ans = 0
        mi, mx = arrays[0][0], arrays[0][-1]
        for arr in arrays[1:]:
            a, b = abs(arr[0] - mx), abs(arr[-1] - mi)
            ans = max(ans, a, b)
            mi = min(mi, arr[0])
            mx = max(mx, arr[-1])
        return ans

**Độ phức tạp:** `O(m)` thời gian (mỗi mảng chỉ nhìn hai đầu mút), `O(1)` bộ nhớ phụ.

### Cách 4 — Giữ hai ứng viên nhỏ nhất / lớn nhất kèm chỉ số mảng

**Ý tưởng khác:** đáp án là `max_{i≠j} (last_j - first_i)`. Gọi `A` là đầu mút trái, `B` là đầu mút phải. Cặp tối ưu thường là `(min A, max B)`, trừ khi chúng thuộc **cùng một mảng**. Khi đó ta phải chuyển sang **ứng viên tốt thứ hai** của một bên. Vậy chỉ cần giữ **hai** đầu mút trái nhỏ nhất và **hai** đầu mút phải lớn nhất (kèm chỉ số mảng), rồi thử cả `2 × 2 = 4` tổ hợp, bỏ tổ hợp trùng chỉ số.

Vì sao hai là đủ? Nếu cặp tốt nhất `(i, j)` có `i = j` thì ta thay bằng "ứng viên thứ hai" ở một phía; ứng viên thứ ba không bao giờ cần vì đã có hai chỉ số khác nhau trong mỗi danh sách.

In [15]:
def max_distance_top2(arrays):
    lows = heapq.nsmallest(2, ((arr[0], i) for i, arr in enumerate(arrays)))     # [(giá trị, chỉ số)]
    highs = heapq.nlargest(2, ((arr[-1], i) for i, arr in enumerate(arrays)))
    best = 0
    for lo, i in lows:
        for hi, j in highs:
            if i != j:
                best = max(best, hi - lo)
    return best


print(max_distance_top2(ex1), max_distance_top2(ex2), max_distance_top2([[1, 5], [2, 3]]))
assert max_distance_top2(ex1) == 4 and max_distance_top2(ex2) == 0 and max_distance_top2([[1, 5], [2, 3]]) == 3

4 0 3


In [16]:
# Code nộp LeetCode cho Cách 4
class SolutionTop2:
    def maxDistance(self, arrays: List[List[int]]) -> int:
        lows = heapq.nsmallest(2, ((arr[0], i) for i, arr in enumerate(arrays)))
        highs = heapq.nlargest(2, ((arr[-1], i) for i, arr in enumerate(arrays)))
        best = 0
        for lo, i in lows:
            for hi, j in highs:
                if i != j:
                    best = max(best, hi - lo)
        return best

**Độ phức tạp:** `O(m)` (`nsmallest(2, ...)` với `k = 2` là hằng số lần `m`), bộ nhớ `O(1)` (lưu 4 phần tử). Cách này tiện khi bạn muốn biết **cặp mảng nào** cho đáp án (xem biến thể ở mục 15).

## 8. TRACE từng bước

Chạy Cách 3 trên `arrays = [[1,2,3],[4,5],[1,2,3]]`.

| bước | mảng `arr` | `arr[-1] - mi` | `mx - arr[0]` | `ans` sau | `mi` sau | `mx` sau |
|---|---|---|---|---|---|---|
| khởi tạo | `[1,2,3]` | — | — | 0 | 1 | 3 |
| 1 | `[4,5]` | `5 - 1 = 4` | `3 - 4 = -1` | 4 | 1 | 5 |
| 2 | `[1,2,3]` | `3 - 1 = 2` | `5 - 1 = 4` | 4 | 1 | 5 |

Kết quả `4`. Hàm trace dưới đây in từng vòng lặp, kể cả ví dụ phản trực giác `[[1,5],[2,3]]` và trường hợp mảng đầu chứa cả min lẫn max toàn cục.

In [17]:
def trace_scan(arrays):
    print("arrays =", arrays)
    mi, mx = arrays[0][0], arrays[0][-1]
    ans = 0
    print(f"khởi tạo: mi={mi}, mx={mx}, ans={ans}")
    for step, arr in enumerate(arrays[1:], 1):
        d1, d2 = arr[-1] - mi, mx - arr[0]
        new_ans = max(ans, d1, d2)
        print(f"bước {step}: arr={arr} | arr[-1]-mi={arr[-1]}-({mi})={d1} | mx-arr[0]={mx}-({arr[0]})={d2} | ans {ans} -> {new_ans}")
        ans = new_ans
        mi, mx = min(mi, arr[0]), max(mx, arr[-1])
        print(f"         cập nhật sau khi so sánh: mi={mi}, mx={mx}")
    print("=> đáp án", ans, "\n")
    return ans


assert trace_scan(ex1) == 4
assert trace_scan([[1, 5], [2, 3]]) == 3
assert trace_scan([[-8, -7, -7, -5, 1, 1, 3, 4], [-2], [-10, -10], [-5, 0, 1, 3, 4, 8]]) == 18

arrays = [[1, 2, 3], [4, 5], [1, 2, 3]]
khởi tạo: mi=1, mx=3, ans=0
bước 1: arr=[4, 5] | arr[-1]-mi=5-(1)=4 | mx-arr[0]=3-(4)=-1 | ans 0 -> 4
         cập nhật sau khi so sánh: mi=1, mx=5
bước 2: arr=[1, 2, 3] | arr[-1]-mi=3-(1)=2 | mx-arr[0]=5-(1)=4 | ans 4 -> 4
         cập nhật sau khi so sánh: mi=1, mx=5
=> đáp án 4 

arrays = [[1, 5], [2, 3]]
khởi tạo: mi=1, mx=5, ans=0
bước 1: arr=[2, 3] | arr[-1]-mi=3-(1)=2 | mx-arr[0]=5-(2)=3 | ans 0 -> 3
         cập nhật sau khi so sánh: mi=1, mx=5
=> đáp án 3 

arrays = [[-8, -7, -7, -5, 1, 1, 3, 4], [-2], [-10, -10], [-5, 0, 1, 3, 4, 8]]
khởi tạo: mi=-8, mx=4, ans=0
bước 1: arr=[-2] | arr[-1]-mi=-2-(-8)=6 | mx-arr[0]=4-(-2)=6 | ans 0 -> 6
         cập nhật sau khi so sánh: mi=-8, mx=4
bước 2: arr=[-10, -10] | arr[-1]-mi=-10-(-8)=-2 | mx-arr[0]=4-(-10)=14 | ans 6 -> 14
         cập nhật sau khi so sánh: mi=-10, mx=4
bước 3: arr=[-5, 0, 1, 3, 4, 8] | arr[-1]-mi=8-(-10)=18 | mx-arr[0]=4-(-5)=9 | ans 14 -> 18
         cập nhật sau khi so sánh: 

**Trace Cách 4** cho `[[1,5],[2,3]]`:

| danh sách | nội dung |
|---|---|
| `lows` (2 đầu trái nhỏ nhất) | `[(1, 0), (2, 1)]` |
| `highs` (2 đầu phải lớn nhất) | `[(5, 0), (3, 1)]` |
| tổ hợp `(lo, hi)` | `(1,5)` bị loại (cùng mảng 0); `(1,3)` → 2; `(2,5)` → 3; `(2,3)` cùng mảng 1 bị loại |

Đáp án `3`. Đây là lý do phải giữ **hai** ứng viên mỗi phía.

In [18]:
def trace_top2(arrays):
    lows = heapq.nsmallest(2, ((arr[0], i) for i, arr in enumerate(arrays)))
    highs = heapq.nlargest(2, ((arr[-1], i) for i, arr in enumerate(arrays)))
    print("lows  =", lows)
    print("highs =", highs)
    best = 0
    for lo, i in lows:
        for hi, j in highs:
            if i == j:
                print(f"  ({lo}@{i}, {hi}@{j}) -> bỏ: cùng một mảng")
            else:
                print(f"  ({lo}@{i}, {hi}@{j}) -> {hi - lo}")
                best = max(best, hi - lo)
    print("=> đáp án", best)
    return best


assert trace_top2([[1, 5], [2, 3]]) == 3

lows  = [(1, 0), (2, 1)]
highs = [(5, 0), (3, 1)]
  (1@0, 5@0) -> bỏ: cùng một mảng
  (1@0, 3@1) -> 2
  (2@1, 5@0) -> 3
  (2@1, 3@1) -> bỏ: cùng một mảng
=> đáp án 3


## 9. Code nộp LeetCode

Chọn **duyệt một lượt** (`O(m)` thời gian, `O(1)` bộ nhớ). Giữ nguyên tên `class Solution`, `maxDistance(self, arrays)`:

In [19]:
from typing import List


class Solution:
    def maxDistance(self, arrays: List[List[int]]) -> int:
        ans = 0
        mi, mx = arrays[0][0], arrays[0][-1]          # tóm tắt các mảng đã duyệt
        for arr in arrays[1:]:
            # so sánh với các mảng TRƯỚC (khác mảng hiện tại)
            ans = max(ans, abs(arr[0] - mx), abs(arr[-1] - mi))
            # rồi mới gộp mảng hiện tại vào tóm tắt
            mi = min(mi, arr[0])
            mx = max(mx, arr[-1])
        return ans


print(Solution().maxDistance([[1, 2, 3], [4, 5], [1, 2, 3]]))   # 4
print(Solution().maxDistance([[1], [1]]))                       # 0
assert Solution().maxDistance(ex1) == 4 and Solution().maxDistance(ex2) == 0

4
0


## 10. Vì sao đúng

**Bước 1: rút gọn về đầu mút.** Với hai mảng khác nhau `i ≠ j` và `x ∈ A_i`, `y ∈ A_j`: vì mảng đã sắp xếp, `first_i ≤ x ≤ last_i` và `first_j ≤ y ≤ last_j`. Do đó `y - x ≤ last_j - first_i` và `x - y ≤ last_i - first_j`. Vậy `|x - y| ≤ max(last_j - first_i, last_i - first_j)`, và dấu bằng đạt khi chọn đúng các đầu mút. Nên khoảng cách tối đa của cặp `(i, j)` đúng bằng `max(last_j - first_i, last_i - first_j)`.

**Bước 2: bỏ `abs`.** Tổng hai biểu thức đó là `(last_i - first_i) + (last_j - first_j) ≥ 0`, nên ít nhất một trong hai `≥ 0`; do đó khoảng cách tối đa toàn bài là `max_{i≠j} (last_j - first_i)`.

**Bước 3: duyệt một lượt phủ hết các cặp.** Với mỗi cặp `(i, j)`, `i < j`, tại thời điểm xử lý mảng `j` thì mảng `i` đã nằm trong `mi, mx`. Ta xét `last_j - mi ≥ last_j - first_i` và `mx - first_j ≥ last_i - first_j`; và mỗi giá trị `last_j - mi`, `mx - first_j` đều là khoảng cách thật của hai mảng **khác nhau** (`mi, mx` đến từ các mảng trước) nên không vượt quá đáp án. Vậy `ans` đúng bằng đáp án.

**Vì sao so sánh trước, cập nhật sau?** Nếu cập nhật `mi, mx` bằng mảng hiện tại trước khi so sánh, thì `arr[-1] - mi` có thể là `arr[-1] - arr[0]` (hai đầu mút của **cùng một mảng**): không hợp lệ.

Ta kiểm chứng tự động: công thức `max_{i≠j}(last_j - first_i)` và cả 4 cách phải trùng oracle vét cạn trên **mọi** input nhỏ (xem mục 12D).

## 11. Time / Space Complexity

Đặt `m` là số mảng, `N` tổng số phần tử.

| Cách | Time | Space |
|---|---|---|
| 1. Vét cạn mọi cặp số | `O(N²)` | `O(1)` |
| 2. Mọi cặp mảng, đầu mút | `O(m²)` | `O(1)` |
| 3. Duyệt một lượt `mi/mx` | `O(m)` | `O(1)` |
| 4. Hai ứng viên + chỉ số | `O(m)` | `O(1)` |

Cách 3 chỉ đọc hai phần tử mỗi mảng nên không phụ thuộc độ dài mảng. Đo thực tế:

In [20]:
def bench(fn, arrays, repeat=3):
    best = 1e9
    for _ in range(repeat):
        t0 = time.perf_counter()
        fn(arrays)
        best = min(best, time.perf_counter() - t0)
    return best


rng = random.Random(11)
mid = random_arrays(rng, 1500, 3, -10000, 10000)          # m = 1500 để cách O(m^2) còn chạy được
print(f"m = {len(mid)}: ")
for name, fn in [("endpoints O(m^2)", max_distance_endpoints), ("scan O(m)", max_distance_scan),
                 ("top2 O(m)", max_distance_top2)]:
    print(f"  {name:18s} {bench(fn, mid) * 1000:9.3f} ms")

big = random_arrays(rng, 10 ** 5, 1, -10000, 10000)       # m = 10^5, mỗi mảng 1 phần tử
for name, fn in [("scan O(m)", max_distance_scan), ("top2 O(m)", max_distance_top2)]:
    print(f"m = 10^5 {name:12s} {bench(fn, big) * 1000:9.3f} ms")

m = 1500: 


  endpoints O(m^2)     128.765 ms
  scan O(m)              0.199 ms
  top2 O(m)              0.214 ms


m = 10^5 scan O(m)       26.150 ms
m = 10^5 top2 O(m)       26.954 ms


## 12. Bộ test

Chia 6 nhóm: (A) ví dụ đề, (B) edge case, (C) phản ví dụ — cài đặt sai phải bị bắt, (D) vét cạn trên mọi input nhỏ, (E) ngẫu nhiên có seed so với oracle, (F) giới hạn lớn nhất.

### A. Ví dụ của đề

In [21]:
IMPLS = {
    "pairs": max_distance_pairs,
    "endpoints": max_distance_endpoints,
    "scan": max_distance_scan,
    "top2": max_distance_top2,
    "formula": max_distance_formula,
    "Solution": lambda a: Solution().maxDistance(a),
    "SolutionEndpoints": lambda a: SolutionEndpoints().maxDistance(a),
    "SolutionScan": lambda a: SolutionScan().maxDistance(a),
    "SolutionTop2": lambda a: SolutionTop2().maxDistance(a),
}


def check_all(arrays, want):
    for name, fn in IMPLS.items():
        assert fn(arrays) == want, (name, arrays, fn(arrays), want)


check_all([[1, 2, 3], [4, 5], [1, 2, 3]], 4)
check_all([[1], [1]], 0)
print("ví dụ 1, 2: cả", len(IMPLS), "cài đặt đúng")

ví dụ 1, 2: cả 9 cài đặt đúng


### B. Edge case

- `m = 2` với mỗi mảng một phần tử.
- Mọi phần tử bằng nhau → `0`.
- Toàn số âm; lẫn âm dương.
- Min và max toàn cục **cùng một mảng** (phải chuyển sang ứng viên thứ hai).
- Min toàn cục ở mảng đầu, max toàn cục ở mảng cuối (và ngược lại).
- Giá trị biên `±10^4`: đáp án `20000`.
- Mảng dài chỉ có hai đầu mút khác biệt.

In [22]:
edge = [
    ([[0], [5]], 5),
    ([[5], [0]], 5),
    ([[3, 3], [3], [3, 3, 3]], 0),
    ([[-5, -3], [-9, -8], [-1]], 8),
    ([[-3, 4], [-2, 2]], 6),                      # min=-3 và max=4 cùng mảng 0 -> chọn (-3,2)=5 hoặc (4,-2)=6
    ([[1, 5], [2, 3]], 3),                        # ví dụ phản trực giác ở mục 4
    ([[1], [2, 3, 4, 5, 6], [7]], 6),
    ([[-10000], [10000]], 20000),
    ([[-10000, 10000], [-10000, 10000]], 20000),
    ([[1, 2, 3, 4, 5, 6, 7, 8, 9, 10]] * 3, 9),
    ([[0, 100], [50], [-100, 0]], 200),
]
for arrays, want in edge:
    check_all(arrays, want)
print(len(edge), "edge case: tất cả cài đặt đúng")

11 edge case: tất cả cài đặt đúng


### C. Phản ví dụ: cài đặt sai phải bị bắt

Một bộ test tốt phải **thất bại** khi code sai. Ta viết 5 bản sai và kiểm tra có ít nhất một input làm chúng khác oracle (chi tiết lỗi ở mục 13).

In [23]:
def bug_global(arrays):                       # max toàn cục - min toàn cục (có thể cùng mảng)
    return max(a[-1] for a in arrays) - min(a[0] for a in arrays)


def bug_update_first(arrays):                 # cập nhật mi/mx TRƯỚC khi so sánh
    mi, mx = arrays[0][0], arrays[0][-1]
    ans = 0
    for arr in arrays[1:]:
        mi = min(mi, arr[0])
        mx = max(mx, arr[-1])
        ans = max(ans, abs(arr[0] - mx), abs(arr[-1] - mi))
    return ans


def bug_one_side(arrays):                     # chỉ xét arr[-1] - mi, quên mx - arr[0]
    mi = arrays[0][0]
    ans = 0
    for arr in arrays[1:]:
        ans = max(ans, arr[-1] - mi)
        mi = min(mi, arr[0])
    return ans


def bug_adjacent(arrays):                     # chỉ so sánh hai mảng liền kề
    return max(max(abs(arrays[i][-1] - arrays[i + 1][0]), abs(arrays[i + 1][-1] - arrays[i][0]))
               for i in range(len(arrays) - 1))


def bug_inf_init(arrays):                     # khởi tạo mi=+inf, mx=-inf rồi so sánh luôn (ghép mảng với chính 'trống')
    mi, mx = float("inf"), float("-inf")
    ans = 0
    for arr in arrays:
        if mx != float("-inf"):
            ans = max(ans, abs(arr[0] - mx), abs(arr[-1] - mi))
        mi = min(mi, arr[0])
        mx = max(mx, arr[-1])
    return ans


BUGS = [bug_global, bug_update_first, bug_one_side, bug_adjacent]
rng = random.Random(1)
tests = [random_arrays(rng, rng.randint(2, 5), 3, -6, 6) for _ in range(400)]
for bug in BUGS:
    bad = next((t for t in tests if bug(t) != max_distance_pairs(t)), None)
    print(f"{bug.__name__:18s} sai trên: {bad} (sai: {bug(bad)}, đúng: {max_distance_pairs(bad)})")
    assert bad is not None
# bug_inf_init thực ra đúng (so sánh chỉ khi đã có mảng trước) -> không phải lỗi, dùng để so
assert all(bug_inf_init(t) == max_distance_pairs(t) for t in tests)
print("bug_inf_init (khởi tạo vô cực, bỏ qua mảng đầu) hoá ra đúng trên", len(tests), "input")

bug_global         sai trên: [[4], [-5, -2, 5]] (sai: 10, đúng: 9)
bug_update_first   sai trên: [[4], [-5, -2, 5]] (sai: 10, đúng: 9)
bug_one_side       sai trên: [[-5, 6, 6], [-5, 1], [1, 4]] (sai: 9, đúng: 11)
bug_adjacent       sai trên: [[0, 2, 5], [-3, -2, 4], [1, 3], [-6, 0, 3]] (sai: 9, đúng: 11)
bug_inf_init (khởi tạo vô cực, bỏ qua mảng đầu) hoá ra đúng trên 400 input


### D. Vét cạn trên mọi input nhỏ

Liệt kê **mọi** bộ mảng: mỗi mảng là dãy tăng (không giảm) có độ dài `1..2` với giá trị trong `{-1, 0, 1, 2}` (14 mảng khả dĩ), với `m = 2` và `m = 3` (`14² + 14³ = 2940` input), và thêm mọi bộ `m = 4` có mảng độ dài `1` (`4⁴ = 256` input). Mọi cài đặt đúng phải trùng oracle `max_distance_pairs` trên tất cả.

In [24]:
vals = [-1, 0, 1, 2]
from itertools import combinations_with_replacement
options = [list(t) for L in (1, 2) for t in combinations_with_replacement(vals, L)]
print("số mảng khả dĩ:", len(options))

count = 0
for m in (2, 3):
    for arrays in product(options, repeat=m):
        arrays = [list(a) for a in arrays]
        want = max_distance_pairs(arrays)
        check_all(arrays, want)
        count += 1
for arrays in product([[v] for v in vals], repeat=4):
    arrays = [list(a) for a in arrays]
    check_all(arrays, max_distance_pairs(arrays))
    count += 1
print(count, "input vét cạn: mọi cài đặt đều khớp oracle")

số mảng khả dĩ: 14
3196 input vét cạn: mọi cài đặt đều khớp oracle


### E. Ngẫu nhiên có seed, đúng ràng buộc

Sinh input ngẫu nhiên (seed cố định) với `m` từ 2 đến 40, độ dài mảng 1..8, giá trị trong khoảng nhỏ (để có nhiều trùng lặp) và khoảng đầy đủ `[-10^4, 10^4]`.

In [25]:
rng = random.Random(20240624)
n_tests = 0
for trial in range(3000):
    m = rng.randint(2, 40)
    span = rng.choice([3, 10, 100, 10000])
    arrays = random_arrays(rng, m, 8, -span, span)
    want = max_distance_pairs(arrays)
    for name, fn in IMPLS.items():
        assert fn(arrays) == want, (name, arrays)
    n_tests += 1
print(n_tests, "input ngẫu nhiên: 9 cài đặt đều đúng")

# tính chất: đáp án <= max toàn cục - min toàn cục, và = nếu min/max toàn cục không cùng mảng
rng = random.Random(3)
for _ in range(500):
    arrays = random_arrays(rng, rng.randint(2, 8), 4, -20, 20)
    ans = Solution().maxDistance(arrays)
    glob = max(a[-1] for a in arrays) - min(a[0] for a in arrays)
    assert 0 <= ans <= glob
    imin = {i for i, a in enumerate(arrays) if a[0] == min(x[0] for x in arrays)}
    imax = {i for i, a in enumerate(arrays) if a[-1] == max(x[-1] for x in arrays)}
    if imin != imax or len(imin) > 1:
        # có một mảng chứa min và một mảng khác chứa max -> đáp án = chênh lệch toàn cục
        if any(i != j for i in imin for j in imax):
            assert ans == glob
print("tính chất 0 <= ans <= global và (khác mảng => ans = global): OK")

3000 input ngẫu nhiên: 9 cài đặt đều đúng
tính chất 0 <= ans <= global và (khác mảng => ans = global): OK


### F. Giới hạn lớn nhất

`m = 10^5` mảng một phần tử; `m = 200` mảng dài `500` (tổng `10^5`); toàn bộ bằng nhau; hai mảng cực đoan ở hai đầu danh sách.

In [26]:
rng = random.Random(77)
big1 = random_arrays(rng, 10 ** 5, 1, -10000, 10000)
big2 = random_arrays(rng, 200, 500, -10000, 10000)
big3 = [[7]] * (10 ** 5)
big4 = [[-10000]] + [[0]] * (10 ** 5 - 2) + [[10000]]

for name, arrays in [("10^5 mảng 1 phần tử", big1), ("200 mảng dài 500", big2), ("toàn số 7", big3), ("cực đoan hai đầu", big4)]:
    t0 = time.perf_counter()
    a1, a2, a3 = Solution().maxDistance(arrays), SolutionTop2().maxDistance(arrays), max_distance_formula(arrays) if len(arrays) <= 1000 else None
    assert a1 == a2
    if a3 is not None:
        assert a1 == a3
    print(f"{name:22s} đáp án = {a1:6d}  ({(time.perf_counter() - t0) * 1000:.1f} ms)")
assert Solution().maxDistance(big3) == 0
assert Solution().maxDistance(big4) == 20000
# đối chiếu big1 bằng công thức O(m) độc lập: max last - min first, trừ trường hợp cùng mảng (mỗi mảng 1 phần tử -> luôn khác mảng nếu khác chỉ số)
vals1 = [a[0] for a in big1]
assert Solution().maxDistance(big1) == max(vals1) - min(vals1)
print("giới hạn lớn nhất OK")

10^5 mảng 1 phần tử    đáp án =  20000  (52.3 ms)
200 mảng dài 500       đáp án =  19999  (3.2 ms)
toàn số 7              đáp án =      0  (38.6 ms)
cực đoan hai đầu       đáp án =  20000  (40.4 ms)
giới hạn lớn nhất OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1: lấy max toàn cục trừ min toàn cục

Nghe rất tự nhiên ("xa nhất = lớn nhất trừ nhỏ nhất") nhưng đề **cấm lấy hai số cùng một mảng**. Khi số nhỏ nhất và số lớn nhất cùng ở một mảng, kết quả bị thổi phồng.

In [27]:
# CODE SAI
def sai1(arrays):
    return max(a[-1] for a in arrays) - min(a[0] for a in arrays)


tc = [[1, 5], [2, 3]]
print("sai1 =", sai1(tc), "| đúng =", Solution().maxDistance(tc))
assert sai1(tc) != Solution().maxDistance(tc)

sai1 = 4 | đúng = 3


**Vì sao sai:** `1` và `5` cùng thuộc mảng 0. **Đúng:** so sánh mảng hiện tại với tóm tắt của các mảng **khác** (mục 9), hoặc dùng hai ứng viên kèm chỉ số (Cách 4).

### Lỗi 2: cập nhật `mi/mx` trước khi so sánh

In [28]:
# CODE SAI
def sai2(arrays):
    mi, mx = arrays[0][0], arrays[0][-1]
    ans = 0
    for arr in arrays[1:]:
        mi = min(mi, arr[0])             # SAI: gộp mảng hiện tại vào trước
        mx = max(mx, arr[-1])
        ans = max(ans, abs(arr[0] - mx), abs(arr[-1] - mi))
    return ans


tc = [[1], [2, 9]]
print("sai2 =", sai2(tc), "| đúng =", Solution().maxDistance(tc))
tc2 = [[5], [1, 100]]
print("sai2 =", sai2(tc2), "| đúng =", Solution().maxDistance(tc2), "(mảng [1,100] tự ghép với chính nó: 99)")
assert sai2(tc2) != Solution().maxDistance(tc2)

sai2 = 8 | đúng = 8
sai2 = 99 | đúng = 95 (mảng [1,100] tự ghép với chính nó: 99)


**Vì sao sai:** sau khi gộp, `mx` có thể là `arr[-1]` và `arr[0]` chính là đầu mút trái của cùng mảng nên ta đang tính `arr[-1] - arr[0]` (cùng mảng). **Đúng:** so sánh rồi mới cập nhật.

### Lỗi 3: chỉ xét một chiều

Chỉ tính `arr[-1] - mi` (phải của mảng hiện tại − trái nhỏ nhất ở trước) nhưng quên `mx - arr[0]`. Sẽ sai khi mảng *trước* có số lớn và mảng hiện tại có số nhỏ.

In [29]:
# CODE SAI
def sai3(arrays):
    mi = arrays[0][0]
    ans = 0
    for arr in arrays[1:]:
        ans = max(ans, arr[-1] - mi)     # SAI: thiếu mx - arr[0]
        mi = min(mi, arr[0])
    return ans


tc = [[10, 20], [1, 2]]
print("sai3 =", sai3(tc), "| đúng =", Solution().maxDistance(tc))
assert sai3(tc) != Solution().maxDistance(tc)

sai3 = 0 | đúng = 19


**Vì sao sai:** `20 - 1 = 19` đạt từ chiều "đầu mút phải của mảng trước − đầu mút trái của mảng sau". **Đúng:** xét **cả hai chiều** `arr[-1] - mi` và `mx - arr[0]` (hoặc dùng `abs`).

### Lỗi 4: chỉ so hai mảng liền kề

"Hai mảng khác nhau" không có nghĩa là liền kề. Mảng đầu và mảng cuối có thể là cặp tối ưu.

In [30]:
# CODE SAI
def sai4(arrays):
    return max(max(abs(arrays[i][-1] - arrays[i + 1][0]), abs(arrays[i + 1][-1] - arrays[i][0]))
               for i in range(len(arrays) - 1))


tc = [[0], [5], [10]]
print("sai4 =", sai4(tc), "| đúng =", Solution().maxDistance(tc), "(0 và 10 nằm ở mảng 0 và 2)")
assert sai4(tc) != Solution().maxDistance(tc)

sai4 = 5 | đúng = 10 (0 và 10 nằm ở mảng 0 và 2)


**Vì sao sai:** bỏ sót cặp `(0, 2)`. **Đúng:** giữ `mi/mx` của **tất cả** mảng đã duyệt.

### Lỗi 5 (bonus): duyệt toàn bộ phần tử để tìm min/max

Không sai nhưng phí: mảng đã sắp xếp nên `arr[0]` và `arr[-1]` đủ. Duyệt cả mảng làm thời gian tăng từ `O(m)` lên `O(N)` — vẫn qua được vì `N ≤ 10^5`, nhưng bỏ qua "dữ kiện đã sắp xếp" của đề là một dấu hiệu chưa đọc kỹ.

## 14. Mẹo & nhận dạng pattern

**Nhận dạng:** đề hỏi `max |a − b|` (hoặc `max (b − a)`) với ràng buộc "hai phần tử thuộc nhóm khác nhau" → thường quy về **max của `phần tử tốt nhất ở nhóm j − phần tử tốt nhất ở nhóm khác`**, giải bằng duyệt một lượt giữ tóm tắt (running min/max) hoặc giữ hai ứng viên tốt nhất.

**Mẹo:**

1. Đọc kỹ "đã sắp xếp": bài này chỉ cần đầu mút.
2. Có điều kiện "khác nhóm" thì dùng **so sánh trước – gộp sau** hoặc **hai ứng viên kèm chỉ số**.
3. Bỏ `abs` bằng cách xét cả hai chiều `max(b - a, a - b)` và để ý khoảng cách luôn không âm.
4. Tự hỏi: "tóm tắt của các phần tử trước là gì?" — thường là `min`, `max`, tổng, đếm.
5. Luôn có oracle vét cạn để so sánh trên input nhỏ, nhất là khi đề có điều kiện phụ như "khác mảng".

**Hay đi cùng pattern:** Best Time to Buy and Sell Stock (121: `max (giá bán − min giá trước đó)`), Maximum Difference Between Increasing Elements (2016), Max Consecutive/Spread problems.

## 15. Biến thể

1. **Trả về cặp chỉ số mảng** `(i, j)` cho đáp án (dùng Cách 4).
2. **Khoảng cách nhỏ nhất** giữa hai số ở hai mảng khác nhau (khó hơn: cần trộn các mảng đã sắp xếp hoặc hai con trỏ).
3. **Mảng chưa sắp xếp:** `min` và `max` của mỗi mảng phải tính bằng `min(arr)`, `max(arr)`: `O(N)`.
4. **Chọn từ `k` mảng khác nhau** để tối đa hoá `max − min` (có thể giải bằng cùng ý tưởng, cần thêm kỹ thuật).
5. **Cho phép chọn cùng mảng**: khi đó đáp án đơn giản là `max toàn cục − min toàn cục`.

Cài đặt biến thể 1 và 5 để so sánh với bài gốc:

In [31]:
def max_distance_with_indices(arrays):
    """Biến thể 1: trả (khoảng cách, i, j) với cặp mảng cho đáp án (lấy first của i, last của j)."""
    lows = heapq.nsmallest(2, ((arr[0], i) for i, arr in enumerate(arrays)))
    highs = heapq.nlargest(2, ((arr[-1], i) for i, arr in enumerate(arrays)))
    best = (0, None, None)
    for lo, i in lows:
        for hi, j in highs:
            if i != j and (best[1] is None or hi - lo > best[0]):
                best = (hi - lo, i, j)
    return best


print(max_distance_with_indices(ex1))
print(max_distance_with_indices([[1, 5], [2, 3]]))
d, i, j = max_distance_with_indices(ex1)
assert d == ex1[j][-1] - ex1[i][0] == Solution().maxDistance(ex1) and i != j


def max_distance_same_allowed(arrays):
    """Biến thể 5: được chọn hai số cùng một mảng -> max toàn cục - min toàn cục."""
    return max(a[-1] for a in arrays) - min(a[0] for a in arrays)


print(max_distance_same_allowed([[1, 5], [2, 3]]), "(khác bài gốc là 3)")

(4, 0, 1)
(3, 1, 0)
4 (khác bài gốc là 3)


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Mọi cặp số | Thử tất cả `x ∈ A_i`, `y ∈ A_j` | `O(N²)` | `O(1)` | Làm oracle để kiểm chứng |
| 2. Mọi cặp mảng, đầu mút | Chỉ xét đầu mút, nhưng mọi `(i, j)` | `O(m²)` | `O(1)` | `m` nhỏ; bước trung gian để hiểu ý tưởng |
| 3. Duyệt một lượt `mi/mx` | So với tóm tắt các mảng trước rồi mới gộp | `O(m)` | `O(1)` | **Đáp án chính** |
| 4. Hai ứng viên + chỉ số | Giữ 2 trái nhỏ nhất, 2 phải lớn nhất, bỏ tổ hợp cùng mảng | `O(m)` | `O(1)` | Khi cần biết cặp mảng; dễ mở rộng cho "k mảng" |

Cả 4 cách cho cùng đáp án trên hơn `3000` input ngẫu nhiên và gần `3200` input vét cạn.

## 17. Bài tập vận dụng

Làm tuần tự từ dễ đến khó. Các cell TODO **chưa được gọi**: tự điền rồi bỏ comment dòng test cuối cell.

### Bài 1 (đoán output)

Đoán trước rồi mới chạy:

In [32]:
arrays = [[-2, 3], [0, 1], [4]]
print([a[0] for a in arrays], [a[-1] for a in arrays])
print(max_distance_scan(arrays))

[-2, 0, 4] [3, 1, 4]
6


### Bài 2 (tính tay)

`arrays = [[1, 4], [0, 5], [3]]`.

- (a) Liệt kê đầu mút trái và phải của từng mảng.
- (b) Cặp `(mảng i, mảng j)` nào cho khoảng cách lớn nhất và bằng bao nhiêu?
- (c) Nếu bỏ ràng buộc "khác mảng", đáp án sẽ là bao nhiêu? Vì sao khác?

Tự giải, rồi kiểm tra ở mục 19.

### Bài 3 (điền TODO)

Hoàn thiện duyệt một lượt:

In [33]:
def max_distance_todo(arrays):
    mi, mx = arrays[0][0], arrays[0][-1]
    ans = 0
    for arr in arrays[1:]:
        # TODO 1: cập nhật ans bằng khoảng cách của arr với mi và mx
        # TODO 2: SAU ĐÓ mới cập nhật mi và mx
        pass
    return ans


# Khi làm xong, bỏ comment để kiểm tra:
# assert max_distance_todo([[1, 2, 3], [4, 5], [1, 2, 3]]) == 4
# assert max_distance_todo([[1, 5], [2, 3]]) == 3

### Bài 4 (tự viết lại)

Không nhìn lời giải, viết lớp `MySolution` dùng duyệt một lượt (hoặc Cách 4):

In [34]:
class MySolution:
    def maxDistance(self, arrays):
        pass            # TODO


# Khi làm xong:
# for a in ([[1, 2, 3], [4, 5], [1, 2, 3]], [[1], [1]], [[1, 5], [2, 3]]):
#     assert MySolution().maxDistance(a) == max_distance_pairs(a)

### Bài 5 (mini challenge): khoảng cách nhỏ nhất giữa hai mảng khác nhau

Viết `min_distance(arrays)` trả về `min |a − b|` với `a, b` thuộc **hai mảng khác nhau** (mảng đã sắp xếp, `m >= 2`). Gợi ý: gộp mọi `(giá trị, chỉ số mảng)` rồi sắp xếp; đáp án nằm giữa hai phần tử **liền kề khác chỉ số mảng**. Hoặc nghĩ vì sao hai phần tử liền kề khác mảng là đủ.

In [35]:
def min_distance(arrays):
    pass            # TODO


# Khi làm xong:
# assert min_distance([[1, 2, 3], [4, 5], [1, 2, 3]]) == 0
# assert min_distance([[1, 10], [4, 12]]) == 2

## 18. Gợi ý 3 mức

**Bài 1**

- Mức 1: nhìn từng mảng, đâu là nhỏ nhất, đâu là lớn nhất?
- Mức 2: `arr[0]` và `arr[-1]` của mỗi mảng.
- Mức 3: trái `[-2, 0, 4]`, phải `[3, 1, 4]`; thử `4 - (-2)` — chúng ở hai mảng khác nhau.

**Bài 2**

- Mức 1: cặp `(i, j)`: `max(last_j - first_i, last_i - first_j)`.
- Mức 2: trái `[1, 0, 3]`, phải `[4, 5, 3]`.
- Mức 3: đầu trái nhỏ nhất `0` (mảng 1), đầu phải lớn nhất `5` (mảng 1): **cùng mảng**. Hãy thử các cặp (mảng 0, mảng 1), (mảng 1, mảng 2), (mảng 0, mảng 2) bằng công thức ở mục 6.

**Bài 3**

- Mức 1: hai khoảng cách: `abs(arr[0] - mx)` và `abs(arr[-1] - mi)`.
- Mức 2: `ans = max(ans, ...)`.
- Mức 3: ba dòng: `ans = max(...)`, `mi = min(mi, arr[0])`, `mx = max(mx, arr[-1])` — theo đúng thứ tự.

**Bài 4**

- Mức 1: tóm tắt các mảng trước là gì?
- Mức 2: `mi`, `mx`.
- Mức 3: xem mục 9.

**Bài 5**

- Mức 1: min khoảng cách giữa hai tập: gộp rồi sắp xếp.
- Mức 2: hai phần tử liền kề trong dãy đã gộp mà thuộc **mảng khác nhau**.
- Mức 3: `merged = sorted((x, i) for i, a in enumerate(arrays) for x in a)`; duyệt `k`, nếu `merged[k][1] != merged[k+1][1]` thì xét `merged[k+1][0] - merged[k][0]`.

## 19. Đáp án tham khảo

Mỗi đáp án đều có `assert` để chạy kiểm tra.

### Bài 1

In [36]:
arrays = [[-2, 3], [0, 1], [4]]
assert [a[0] for a in arrays] == [-2, 0, 4] and [a[-1] for a in arrays] == [3, 1, 4]
assert max_distance_scan(arrays) == 6 == max_distance_pairs(arrays)
print("Bài 1: 6 (từ -2 ở mảng 0 và 4 ở mảng 2)")

Bài 1: 6 (từ -2 ở mảng 0 và 4 ở mảng 2)


### Bài 2

(a) trái `[1, 0, 3]`, phải `[4, 5, 3]`. (b) Cặp mảng `(0, 1)`: `max(5 - 1, 4 - 0) = 4`; cặp `(1, 2)`: `max(3 - 0, 5 - 3) = 3`; cặp `(0, 2)`: `max(3 - 1, 4 - 3) = 2`. Lớn nhất là `4`. (c) Bỏ ràng buộc: `max toàn cục − min toàn cục = 5 − 0 = 5`, vì cả `0` và `5` cùng thuộc mảng 1.

In [37]:
arrays = [[1, 4], [0, 5], [3]]
assert max_distance_pairs(arrays) == 4
assert max_distance_same_allowed(arrays) == 5
for name, fn in IMPLS.items():
    assert fn(arrays) == 4, name
print("Bài 2 OK")

Bài 2 OK


### Bài 3

In [38]:
def max_distance_answer(arrays):
    mi, mx = arrays[0][0], arrays[0][-1]
    ans = 0
    for arr in arrays[1:]:
        ans = max(ans, abs(arr[0] - mx), abs(arr[-1] - mi))
        mi = min(mi, arr[0])
        mx = max(mx, arr[-1])
    return ans


assert max_distance_answer([[1, 2, 3], [4, 5], [1, 2, 3]]) == 4
assert max_distance_answer([[1, 5], [2, 3]]) == 3
print("Bài 3 OK")

Bài 3 OK


### Bài 4

In [39]:
class MySolutionAnswer:
    def maxDistance(self, arrays):
        mi, mx = arrays[0][0], arrays[0][-1]
        best = 0
        for arr in arrays[1:]:
            best = max(best, arr[-1] - mi, mx - arr[0])
            mi, mx = min(mi, arr[0]), max(mx, arr[-1])
        return best


rng = random.Random(4)
for _ in range(300):
    a = random_arrays(rng, rng.randint(2, 6), 4, -15, 15)
    assert MySolutionAnswer().maxDistance(a) == max_distance_pairs(a)
print("Bài 4 OK")

Bài 4 OK


### Bài 5

Hai phần tử gần nhất thuộc hai mảng khác nhau phải liền kề trong dãy gộp: nếu giữa chúng còn phần tử `z` khác, thì `z` thuộc mảng khác với ít nhất một trong hai, và cặp `(z, một trong hai)` cho khoảng cách không lớn hơn.

In [40]:
def min_distance_answer(arrays):
    merged = sorted((x, i) for i, a in enumerate(arrays) for x in a)
    best = float("inf")
    for k in range(len(merged) - 1):
        if merged[k][1] != merged[k + 1][1]:
            best = min(best, merged[k + 1][0] - merged[k][0])
    return best


def min_distance_oracle(arrays):
    return min(abs(x - y) for i, j in combinations(range(len(arrays)), 2) for x in arrays[i] for y in arrays[j])


assert min_distance_answer([[1, 2, 3], [4, 5], [1, 2, 3]]) == 0
assert min_distance_answer([[1, 10], [4, 12]]) == 2
rng = random.Random(55)
for _ in range(500):
    a = random_arrays(rng, rng.randint(2, 6), 4, -30, 30)
    assert min_distance_answer(a) == min_distance_oracle(a), a
print("Bài 5 OK")

Bài 5 OK


## 20. Template nhớ nhanh + Checklist trước khi submit

**Template (tối ưu giữa các nhóm khác nhau, duyệt một lượt):**

```python
class Solution:
    def maxDistance(self, arrays):
        mi, mx = arrays[0][0], arrays[0][-1]       # tóm tắt các mảng đã duyệt
        ans = 0
        for arr in arrays[1:]:
            ans = max(ans, abs(arr[0] - mx), abs(arr[-1] - mi))   # so sánh TRƯỚC
            mi = min(mi, arr[0])                                  # gộp SAU
            mx = max(mx, arr[-1])
        return ans
```

**Checklist trước khi submit:**

- [ ] Đã dùng việc "mảng sắp xếp" để chỉ nhìn `arr[0]` và `arr[-1]`?
- [ ] So sánh mảng hiện tại với `mi/mx` **trước** khi cập nhật `mi/mx`?
- [ ] Xét **cả hai chiều** (`arr[-1] - mi` và `mx - arr[0]`)?
- [ ] Không dùng `max toàn cục − min toàn cục` (có thể cùng mảng)?
- [ ] Mảng đầu được dùng để khởi tạo; vòng lặp bắt đầu từ mảng thứ hai?
- [ ] Đã thử `[[1,5],[2,3]]` (đáp án 3), `[[1],[1]]` (0), `[[0],[5],[10]]` (10)?
- [ ] Đúng tên `class Solution`, `maxDistance(self, arrays)`, không in debug.